# 12 Explainable Fraud Detection (XAI) & Graph Evidence
## Local & Global Interpretability via SHAP and Knowledge Graph Topology

**Project Scope:** Academic & Research Pipeline  
**Models Explained:** Supervised XGBoost Pipeline (`models/fraud_model/model.joblib`)  
**Graph Context:** Heterogeneous Insurance Knowledge Graph (`data/graph/`, `data/features/graph_features.csv`)  

---

### Research Objectives
1. **Global Model Interpretability:** Compute SHAP (SHapley Additive exPlanations) values to rank global drivers of fraud predictions.
2. **Local Factor Decomposition:** For any individual claim, decompose predictions into:
   - **Top Risk-Increasing Factors** (features pushing probability towards fraud)
   - **Top Risk-Decreasing Factors** (features supporting legitimacy)
3. **Graph Evidence Synthesis:** Ground explanations in verifiable graph evidence:
   - *Suspicious Connections*
   - *High-Degree Entities*
   - *Repeated Relationships*
   - *Neighboring Flagged Claims*
4. **Human-in-the-Loop Reporting:** Translate mathematical Shapley values and topological metrics into clear, non-technical natural language summaries.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import shap
import joblib

from src.explainability.shap_explainer import ShapExplainer
from src.explainability.graph_explainer import GraphExplainer
from src.explainability.claim_explainer import ClaimExplainer, explain_claim

print("Explainability engine initialized successfully.")

## 1. Load Trained Supervised Model & Feature Dataset

In [2]:
features_df = pd.read_csv('../data/features/final_claim_features.csv')
model_pipeline = joblib.load('../models/fraud_model/model.joblib')
feature_cols = list(model_pipeline.feature_names_in_)

X = features_df[feature_cols]
print(f"Loaded {len(features_df)} claims across {len(feature_cols)} input features.")

## 2. Global SHAP Feature Importance
We use `TreeExplainer` on the transformed feature matrix to compute global mean absolute Shapley values.

In [3]:
preprocessor = model_pipeline.named_steps['preprocessor']
classifier = model_pipeline.named_steps['classifier']

X_trans = preprocessor.transform(X)
transformed_names = list(preprocessor.get_feature_names_out())

explainer = shap.TreeExplainer(classifier)
shap_values = explainer(X_trans)

mean_abs_shap = np.mean(np.abs(shap_values.values), axis=0)
importance_df = pd.DataFrame({
    'feature': transformed_names,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print("Top 10 Global Features by Mean Absolute SHAP Impact:")
display(importance_df.head(10))

## 3. Local Claim Explanation (High-Risk Fraud Claim: CLM00001)
Inspect local feature attributions partitioned into risk-increasing vs risk-decreasing.

In [4]:
claim_explainer = ClaimExplainer()
exp_high_risk = claim_explainer.explain_claim('CLM00001', top_k=5)

print(f"Claim ID: {exp_high_risk['claim_id']}")
print(f"Fraud Probability: {exp_high_risk['fraud_probability'] * 100:.2f}%")
print(f"Base Expected Value: {exp_high_risk['base_value']:.4f}")

print("\n--- Top Risk-Increasing Factors (+ impact) ---")
for f in exp_high_risk['top_positive_factors']:
    print(f"  [+] {f['feature']}: +{f['impact']:.4f}")

print("\n--- Top Risk-Decreasing / Mitigating Factors (- impact) ---")
for f in exp_high_risk['top_negative_factors']:
    print(f"  [-] {f['feature']}: {f['impact']:.4f}")

## 4. Graph Evidence Extraction for High-Risk Claim

In [5]:
graph_exp = exp_high_risk['graph_explanation']

print("--- Suspicious Connections ---")
for sc in graph_exp['suspicious_connections']:
    print(f"  * {sc['type']}: {sc['description']}")

print("\n--- High-Degree Entities in Ego-Network ---")
for hd in graph_exp['high_degree_entities']:
    print(f"  * {hd['entity_type']} {hd['name']} (degree: {hd['degree']}, claims: {hd['claim_count']})")

print("\n--- Repeated Relationships ---")
for rr in graph_exp['repeated_relationships']:
    print(f"  * {rr['relationship_type']}: {rr['description']}")

print("\n--- Neighboring Flagged Claims ---")
for nc in graph_exp['neighboring_flagged_claims']:
    print(f"  * {nc['claim_id']}: {nc['description']}")

## 5. Local Claim Explanation (Legitimate Baseline Claim: CLM00002)

In [6]:
exp_low_risk = claim_explainer.explain_claim('CLM00002', top_k=5)

print(f"Claim ID: {exp_low_risk['claim_id']}")
print(f"Fraud Probability: {exp_low_risk['fraud_probability'] * 100:.2f}%")
print("\nTop Factors:")
for f in exp_low_risk['top_factors']:
    print(f"  {f['direction'].upper()}: {f['feature']} ({f['impact']:+.4f})")

print("\nPlain-English Investigator Narrative:")
print(exp_low_risk['summary_text'])

## 6. Academic Research Summary & Evidence Verification
- **SHAP TreeExplainer:** Solves exact game-theoretic cooperative Shapley values on XGBoost tree leaves, ensuring local accuracy and consistency.
- **Graph Evidence Grounding:** Rather than asserting generic collusion, the explainer references exact relational edges, repeated provider-claimant interactions, and flagged neighbor claim IDs.
- **Zero Fabricated Insights:** All explanation components correspond to reproducible mathematical and topological derivations.